In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 17 — CLASSES DESBALANCEADAS: QUANDO A MINORIA IMPORTA MAIS

> "Em uma democracia, a maioria vence. Em um dataset, a maioria pode enganar. A verdadeira inteligência está em dar voz à minoria silenciosa e crucial."
> — Um Cientista de Dados de um campo de nicho

Eu havia ajustado a sensibilidade do modelo pelo *threshold*. Mas uma questão sobre a natureza dos dados ainda me incomodava. Meu treino tinha 304 benignos e 176 malignos — quase 2 para 1. Um **desbalanceamento** que, durante o próprio *treinamento*, poderia enviesar o modelo: ele pode ficar "preguiçoso", percebendo que acerta muito só favorecendo a maioria.

Eu já lidara com isso na avaliação (métricas, *threshold*). Mas não atacara a raiz. Eu queria garantir que a voz da classe maligna — a que mais importa — fosse ouvida com a mesma clareza durante o aprendizado. Não podia inventar casos de câncer, mas podia reequilibrar a balança.

## Reequilibrando a Balança

Duas famílias de técnicas atacam o desbalanceamento:

**Ajuste de peso (`class_weight`)** — não altera os dados; muda a função de custo do treino, penalizando mais os erros na classe minoritária. No *scikit-learn*, `class_weight="balanced"` faz isso automaticamente. É simples e "de graça".

**Reamostragem (SMOTE)** — altera os dados de treino. Em vez de copiar exemplos da minoria, o **SMOTE** cria exemplos *sintéticos* por interpolação entre vizinhos. Deve ficar **dentro do *pipeline*** para ser aplicado só ao treino de cada *fold* (senão, *leakage*).

Comparamos ambas contra o modelo afinado, medindo o que de fato importa: o **recall da classe maligna**.

#### Código 17.1: Ajuste de Peso das Classes


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

metricas = {"acuracia": "accuracy", "recall_Maligno": recall_maligno}

# class_weight="balanced" pesa mais os erros na classe minoritaria (maligna).
com_peso = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                     ("escala", StandardScaler()),
                     ("svm", SVC(kernel="rbf", C=10, gamma=0.01,
                                 class_weight="balanced", random_state=42))])

r = cross_validate(com_peso, X_train, y_train, cv=5, scoring=metricas)
print(f"class_weight=balanced -> acuracia {r['test_acuracia'].mean():.4f} | "
      f"recall Maligno {r['test_recall_Maligno'].mean():.4f}")

class_weight=balanced -> acuracia 0.9750 | recall Maligno 0.9605


#### Código 17.2: SMOTE — Exemplos Sintéticos da Minoria


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# O pipeline do imblearn permite passos de reamostragem no fluxo da CV.
com_smote = ImbPipeline([("selecao", SelectKBest(info_mutua, k=25)),
                         ("escala", StandardScaler()),
                         ("smote", SMOTE(random_state=42)),
                         ("svm", SVC(kernel="rbf", C=10, gamma=0.01, random_state=42))])

r = cross_validate(com_smote, X_train, y_train, cv=5, scoring=metricas)
print(f"SMOTE -> acuracia {r['test_acuracia'].mean():.4f} | "
      f"recall Maligno {r['test_recall_Maligno'].mean():.4f}")

SMOTE -> acuracia 0.9750 | recall Maligno 0.9548


O modelo afinado, **sem** tratamento, tinha acurácia 97,71% e recall maligno **94,90%**. Os dois métodos trocam uma fração mínima de acurácia (→ 97,50%) por mais recall na classe que importa: **`class_weight` sobe o recall maligno para 96,05%** e o SMOTE para 95,48%. Aqui, o vencedor é o mais simples: `class_weight` entrega o melhor recall maligno **sem custo computacional** e sem criar dados artificiais.

> **⚠️ ATENÇÃO — Pacientes sintéticos exigem cautela**
>
> O SMOTE cria "pacientes" por interpolação matemática entre casos reais. Isso é poderoso, mas num contexto clínico pede prudência: um paciente sintético é uma combinação de dois reais, **não** um caso clínico validado. Quando o ganho do SMOTE não for claramente superior — como aqui — prefira a solução mais simples e transparente (`class_weight`).

> **📌 CHECKLIST DO CAPÍTULO 17**
>
> ✅ Entende por que o desbalanceamento pode enviesar o *treinamento*.
>
> ✅ Conhece `class_weight="balanced"` e o **SMOTE**, e sabe manter o SMOTE dentro do *pipeline*.
>
> ✅ Executou os Códigos 17.1 e 17.2 e comparou pelo **recall da classe maligna** (não pela média): `class_weight` 96,05% vs SMOTE 95,48% vs 94,90% sem tratamento.
>
> **⚠️ CRÍTICO:** Sempre que o dataset for desbalanceado, considere `class_weight` ou SMOTE — mas escolha pela métrica que importa (aqui, recall maligno), não pela acurácia global.

A balança estava equilibrada. Combinando o ajuste de peso com o *threshold* do capítulo anterior, eu tinha um modelo não apenas preciso, mas **sensível** à classe que mais importava. Eu abordara a avaliação de vários ângulos: dissecando erros, ajustando a sensibilidade, corrigindo o viés do treino.

Mas uma pergunta ainda pairava, a mesma desde o início: *por quê?* Eu via *o que* o modelo fazia, não *como* ele pensava. Para apresentá-lo ao chefe da oncologia, eu precisaria mais que matrizes e curvas. Precisaria abrir a caixa-preta. Era hora de mergulhar na interpretabilidade.

**PARTE VI — INTERPRETABILIDADE E CONFIANÇA**
